# Features finales

Aplica la lista de features del punto 3 de `docs/decisions.md` sobre la foto de `03_dataset.ipynb` (una fila por merchant).

- **24 features:** 21 originales + 3 relativas a la industria.
- Sin lags ni diferencias: las ventanas de 7, 15 y 30 días ya están en la foto.

En las tasas, el valor −1 (sin transacciones) se toma como nulo. `group_d` se conserva para la sensibilidad y no entra al modelo.

Salida: `data/processed/features.parquet` con `merchant_id`, `observation_date`, `target`, `group_d` y 24 features.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
MERCHANT_PATH = ROOT / "data" / "merchant_rating_featureset_sample.parquet"
DATASET_PATH = ROOT / "data" / "processed" / "dataset.parquet"
OUT_PATH = ROOT / "data" / "processed" / "features.parquet"
KEYS = ["merchant_id", "observation_date"]

BASE_FEATURES = [
    # aprobación actual
    "me_txn_approval_rate_1d", "me_txn_approval_rate_7d", "me_txn_approval_rate_15d", "me_txn_approval_rate_30d",
    "me_tpv_approval_rate_1d", "me_tpv_approval_rate_7d", "me_tpv_approval_rate_15d", "me_tpv_approval_rate_30d",
    # aprobación histórica
    "me_lifetime_txn_approval_rate", "me_lifetime_tpv_approval_rate",
    # ticket
    "me_attempted_tpv_avg_1d", "me_attempted_tpv_avg_7d", "me_attempted_tpv_avg_30d", "me_lifetime_tpv_avg",
    # intentos
    "me_attempted_txn_sum_1d", "me_attempted_txn_sum_7d", "me_attempted_txn_sum_15d", "me_attempted_txn_sum_30d",
    # ecommerce y tamaño
    "me_attempted_ecommerce_txn_sum_7d", "approved_tpv_90d", "me_lifetime_approved_txn",
]
INDUSTRY_SOURCES = ["approved_txns_90d", "p50_approved_tpv_90d", "p95_approved_tpv_90d", "p50_approved_txns_90d"]

## Foto del día elegido

In [ ]:
dataset = pd.read_parquet(DATASET_PATH, columns=KEYS + ["target", "group_d"])
dataset["observation_date"] = pd.to_datetime(dataset["observation_date"])
dates = sorted(dataset["observation_date"].dt.date.unique())

daily = pd.read_parquet(
    MERCHANT_PATH,
    columns=KEYS + BASE_FEATURES + INDUSTRY_SOURCES,
    filters=[("observation_date", "in", dates)],
)
daily["observation_date"] = pd.to_datetime(daily["observation_date"])
rate_cols = [c for c in daily.columns if "rate" in c]
daily[rate_cols] = daily[rate_cols].where(daily[rate_cols] > -0.5)  # -1 = sin transacciones

features = dataset.merge(daily, on=KEYS, how="left")
print(f"{len(features):,} filas")

## Relativas a la industria

In [ ]:
def ratio(num, den):
    return (num / den.where(den > 0)).astype("float64")


features["tpv_vs_industry_p50"] = ratio(features["approved_tpv_90d"], features["p50_approved_tpv_90d"])
features["tpv_vs_industry_p95"] = ratio(features["approved_tpv_90d"], features["p95_approved_tpv_90d"])
features["txns_vs_industry_p50"] = ratio(features["approved_txns_90d"], features["p50_approved_txns_90d"])
features = features.drop(columns=INDUSTRY_SOURCES)

## Conteo

In [ ]:
feature_cols = [c for c in features.columns if c not in KEYS + ["target", "group_d"]]
print(f"{len(feature_cols)} features")
assert len(feature_cols) == 24
assert len(features) == len(dataset)

In [ ]:
nulls = features[feature_cols].isna().mean().sort_values(ascending=False)
print("nulos por feature (las 15 con más):")
print((nulls.head(15) * 100).round(1).to_string())

OUT_PATH.parent.mkdir(exist_ok=True)
features.to_parquet(OUT_PATH, index=False)
print("guardado", OUT_PATH)